# Audit exploratoire — Amazon Reviews 2023

## Pourquoi ce notebook existe

Ce notebook est le **compagnon interactif** de la chaîne d'audit `src/data/audit/01..04_*.py`. Sa raison d'être :

- Les 4 scripts produisent du **JSON** + **PNG** + un **rapport markdown** committables et reproductibles. C'est la source de vérité.
- Le notebook permet, en parallèle, d'**explorer librement** le dataset (tester une nouvelle requête polars, regarder un échantillon de lignes, vérifier visuellement une distribution surprenante) sans casser le pipeline officiel.
- Toute observation d'intérêt qui sort du notebook **doit être remontée dans le rapport** `reports/audit_v1_amazon_reviews_2023.md` (le notebook n'est pas le livrable, le rapport l'est).

## Place dans le projet

```
Phase P02 — Données comme matière première
└── Sous-todo 1.1 — Audit dataset brut  (← TU ES ICI)
    ├── Scripts 01..04          [code Python officiel]
    ├── reports/audit_v1_*.md   [synthèse humaine 200+ lignes]
    ├── reports/audit_*.json    [métriques machine-readable]
    └── ce notebook             [exploration interactive en pandas]

→ Sous-todo 1.2 — Cleaning + feature engineering + split (à venir, C06)
→ Sous-todo 1.3 — Anti-leakage : 5 patterns (à venir, C07)
→ Sous-todo 1.4 — Validation Pandera (à venir, C08)
```

## Discipline (rappel des invariants C05)

1. **Audit décrit, cleaning corrige.** On ne modifie **pas** les fichiers `data/raw/full/` depuis ce notebook. Toute correction part en C06.
2. **Pas de chiffre sans source.** Tout chiffre cité dans le rapport doit venir d'un JSON métriques généré par `make audit`, pas d'une cellule du notebook (les cellules sont éphémères).
3. **Reproductibilité.** Les 4 scripts officiels utilisent `seed=42` et un schéma fixe. Le notebook ici est volontairement plus libre pour permettre l'exploration.

## État du périmètre (à valider)

Ce notebook lit **3 catégories complètes** : Electronics, Toys_and_Games, All_Beauty. Ce périmètre a été choisi initialement par diversité supposée — un **biais de supposition** qu'on est en train de corriger via la sous-étape "meta-first" (cf. décision **D-008** à venir, qui s'appuiera sur `reports/00_dataset_meta_exploration.md` plutôt que sur l'intuition).

Donc : si tu lis ce notebook après que D-008 ait étendu le périmètre, mets à jour la liste `CATEGORIES` dans `01_load_full.py` et relance `make audit-load`.

## Comment exécuter

Préalable : `make install-data` puis `make audit-load` doivent avoir tourné (les parquets doivent exister dans `data/raw/full/`).

Puis ouvrir ce notebook avec `jupyter lab notebooks/` ou via VSCode (extension Jupyter), et exécuter cellule par cellule.

## Pourquoi pandas ici (et polars dans les scripts)

Les **scripts d'audit officiels** utilisent **polars lazy** car ils opèrent sur 60+ M lignes (35 GB JSONL → 8,8 GB parquet). À ce volume, pandas en RAM ne tient pas.

Ce **notebook** travaille sur des **échantillons** (lecture lazy + `.head()`, `.sample()`, ou filtres ciblés). À ce niveau-là (< 1 GB en RAM), pandas est plus ergonomique pour l'exploration interactive (jolies tables Jupyter, `.describe()`, `df.plot()` une ligne…). On va donc faire un mix : **polars** pour les agrégations qui touchent toutes les lignes, **pandas** pour les explorations qui ne portent que sur quelques milliers de lignes.

Décision tracée : `BRAIN/decisions.md` D-006.

---

## Setup — imports et chemins

On importe polars (moteur lourd), pandas (interop notebook), matplotlib + seaborn (graphiques). On résout le `REPO_ROOT` de manière à ce que le notebook fonctionne qu'on l'ouvre depuis `notebooks/` ou depuis la racine du repo.

In [ ]:
from pathlib import Path

import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
REVIEWS_DIR = REPO_ROOT / 'data' / 'raw' / 'full' / 'reviews'
META_DIR = REPO_ROOT / 'data' / 'raw' / 'full' / 'meta'
REPORTS_DIR = REPO_ROOT / 'reports'

sns.set_theme(style='whitegrid', context='notebook')

assert REVIEWS_DIR.exists() and any(REVIEWS_DIR.iterdir()), (
    'Les parquets reviews ne sont pas téléchargés. Lance `make audit-load` depuis la racine.'
)

print('Repo root :', REPO_ROOT)
print('Reviews   :', sorted(p.name for p in REVIEWS_DIR.iterdir()))
print('Meta      :', sorted(p.name for p in META_DIR.iterdir()))

---

## Section 1 — Charger les parquets via polars (lazy)

**Ce qu'on fait** : ouvrir les 6 parquets (3 reviews + 3 meta) en mode **lazy** avec `polars.scan_parquet()`. Aucune donnée n'est chargée en RAM à ce stade — polars construit juste un plan d'exécution.

**Pourquoi lazy et pas eager** : les fichiers reviews font jusqu'à 5,4 GB chacun (Electronics). Les charger en RAM les uns après les autres saturerait la mémoire. Le mode lazy permet de filtrer/agréger d'abord, puis ne matérialiser que le résultat (typiquement quelques milliers de lignes).

**Ce qu'on cherche à valider** : que les 6 parquets existent, ont des colonnes attendues, et que la concat lazy fonctionne.

In [ ]:
CATEGORIES = ('Electronics', 'Toys_and_Games', 'All_Beauty')


def scan_concat(dir_path: Path) -> pl.LazyFrame:
    """Concat lazy des parquets d'une famille (reviews ou meta), avec colonne _source_category."""
    paths = [dir_path / f'{cat}.parquet' for cat in CATEGORIES]
    return pl.concat(
        [pl.scan_parquet(p).with_columns(pl.lit(cat).alias('_source_category'))
         for p, cat in zip(paths, CATEGORIES, strict=True)],
        how='diagonal_relaxed',
    )

reviews_lf = scan_concat(REVIEWS_DIR)
meta_lf = scan_concat(META_DIR)

print('Schéma reviews :')
print(reviews_lf.collect_schema())
print()
print('Schéma meta :')
print(meta_lf.collect_schema())

## Section 2 — Volumes : combien de lignes a-t-on vraiment ?

**Ce qu'on fait** : matérialiser uniquement les **comptes** (pas les données). Polars exécute le plan complet sur disque columnar mais ne ramène en RAM que des entiers.

**Ce qu'on cherche** : confirmer les volumes officiels mesurés par les scripts (60,85 M reviews + 2,61 M items sur les 3 cat). Si les chiffres divergent, c'est qu'un parquet a été corrompu ou tronqué — à investiguer.

In [ ]:
n_reviews = reviews_lf.select(pl.len()).collect().item()
n_meta = meta_lf.select(pl.len()).collect().item()

per_cat_reviews = (
    reviews_lf.group_by('_source_category')
    .agg(pl.len().alias('n'))
    .sort('n', descending=True)
    .collect()
)

per_cat_meta = (
    meta_lf.group_by('_source_category')
    .agg(pl.len().alias('n'))
    .sort('n', descending=True)
    .collect()
)

print(f'Total reviews : {n_reviews:_}')
print(f'Total meta    : {n_meta:_}')
print()
print('Reviews par catégorie :')
print(per_cat_reviews)
print()
print('Meta par catégorie :')
print(per_cat_meta)

## Section 3 — Audit qualité (NaN, doublons)

**Ce qu'on fait** : recalculer les ratios de NaN par colonne et le nombre de `parent_asin` uniques sur l'ensemble du dataset. Comparaison avec ce qu'a calculé `02_audit_qualite.py`.

**Ce qu'on cherche** :
- Confirmer que les colonnes reviews sont à 0 % NaN (dataset propre)
- Confirmer la concentration NaN sur metadata : `bought_together`, `author`, `subtitle` (≥ 99 %)
- Confirmer le duplication factor sur `parent_asin` côté reviews (~23, mesuré par les scripts)

Si une mesure ici diffère du JSON officiel, c'est que l'un des deux a un bug — à corriger avant cleaning.

In [ ]:
# NaN ratio reviews
schema_reviews = reviews_lf.collect_schema().names()
nan_reviews = reviews_lf.select(
    [pl.col(c).is_null().mean().alias(c) for c in schema_reviews]
).collect().to_dicts()[0]

pd.Series(nan_reviews).sort_values(ascending=False).head(15).to_frame('nan_ratio_reviews')

In [ ]:
# NaN ratio meta
schema_meta = meta_lf.collect_schema().names()
nan_meta = meta_lf.select(
    [pl.col(c).is_null().mean().alias(c) for c in schema_meta]
).collect().to_dicts()[0]

pd.Series(nan_meta).sort_values(ascending=False).head(20).to_frame('nan_ratio_meta')

In [ ]:
# Duplication par parent_asin
uniqueness = reviews_lf.select(
    n_total=pl.len(),
    n_unique_parent_asin=pl.col('parent_asin').n_unique(),
    n_unique_user_id=pl.col('user_id').n_unique(),
    n_unique_asin=pl.col('asin').n_unique(),
).collect().to_dicts()[0]

duplication_factor = uniqueness['n_total'] / uniqueness['n_unique_parent_asin']
print(f"Duplication factor parent_asin = {duplication_factor:.2f} (attendu ~23 d'après le rapport)")
uniqueness

## Section 4 — Distributions visuelles

**Ce qu'on fait** : tracer rapidement les 4 distributions clés (catégorielle, ratings, longueurs textes, prix). Les graphiques officiels sont déjà sauvés dans `reports/figures/`, donc ici on est en mode **vérification visuelle rapide** : un pic suspect ou un trou bizarre nous alerte.

**Différence avec les scripts** : ici on plot dans le notebook (interactif), là-bas on sauve en PNG (commitable). Même chiffres, livrables différents.

In [ ]:
# Distribution catégorielle reviews
fig, ax = plt.subplots(figsize=(8, 4))
per_cat_reviews.to_pandas().set_index('_source_category')['n'].plot(
    kind='bar', ax=ax, color='#2563eb'
)
ax.set_title('Reviews par catégorie (FULL)')
ax.set_ylabel('Nombre de reviews')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution des ratings — calcul via polars puis plot via pandas
ratings = (
    reviews_lf.group_by(
        pl.col('rating').cast(pl.Float64, strict=False).round(0).cast(pl.Int64).alias('r')
    )
    .agg(pl.len().alias('n'))
    .sort('r')
    .collect()
)

fig, ax = plt.subplots(figsize=(7, 4))
ratings.to_pandas().set_index('r')['n'].plot(
    kind='bar', ax=ax, color='#f59e0b'
)
ax.set_title('Distribution des ratings (FULL, 60M+ reviews)')
ax.set_xlabel('Note (1-5)')
plt.tight_layout()
plt.show()

## Section 5 — Exploration libre

**Ce qu'on fait** : sortir un échantillon de lignes pour le voir "de ses yeux". Toutes les statistiques agrégées du monde ne remplacent pas le réflexe de regarder 5 lignes au hasard.

**Ce qu'on cherche** : du bizarre. Des champs qui ressemblent pas à ce que la doc HF promet. Des encodages cassés (mojibake), des HTML non échappés, des langues inattendues, etc. Si tu trouves quelque chose d'intéressant, **note-le** dans `reports/audit_v1_amazon_reviews_2023.md` section 9 (Annexes) ou `BRAIN/learnings.md`.

In [ ]:
# Échantillon aléatoire reviews — collect en pandas pour l'affichage Jupyter
sample = (
    reviews_lf.select(
        'rating', 'title', 'text', 'parent_asin', 'verified_purchase',
        'helpful_vote', '_source_category',
    )
    .collect()
    .sample(n=5, seed=42)
    .to_pandas()
)
sample

In [ ]:
# Échantillon aléatoire meta — pour voir à quoi ressemblent les descriptions produit
sample_meta = (
    meta_lf.select(
        'main_category', 'title', 'price', 'description', 'parent_asin',
        '_source_category',
    )
    .collect()
    .sample(n=5, seed=42)
    .to_pandas()
)
sample_meta

## Section 6 — Pour aller plus loin

Le notebook s'arrête ici délibérément. Les analyses **canoniques** (biais, leakages, calculs de prix robustes) sont dans les scripts officiels :

- `make audit-quality` → `reports/audit_qualite_metrics.json` + figures NaN
- `make audit-dist` → `reports/audit_distributions_metrics.json` + figures distributions
- `make audit-bias` → `reports/audit_biais_leakage_metrics.json`
- `make audit` → enchaîne les trois

**Synthèse humaine** : `reports/audit_v1_amazon_reviews_2023.md` (verdict + conditions à respecter en C06/C07)

**Décisions tracées** : `BRAIN/decisions.md` (D-004 à D-007 et au-delà)

Si tu utilises ce notebook pour creuser une question précise (ex : *"pourquoi y a-t-il 1 G d'écart entre n_unique_user_id et n_total_reviews ?"*), termine ta session par :

1. Un paragraphe de conclusion dans le notebook (cellule markdown finale).
2. Une remontée de la conclusion dans le rapport markdown ou dans `BRAIN/learnings.md`.
3. Pas de `df.to_parquet(...)` — l'audit ne modifie pas les données. Si tu veux corriger quelque chose, c'est en C06.